# 12. Análisis final de experimentos

Este notebook cierra la etapa experimental del proyecto mediante:

1. Validación de los resultados reales de GLM, MLP y PINN.
2. Comparaciones estadísticas pareadas.
3. Análisis de ganadores y brecha de generalización.
4. Relación exploratoria entre repetibilidad de los bloques y error.
5. Consistencia bidireccional de parámetros y HRF de la PINN.
6. Exportación de tablas y figuras listas para el informe.

> **Nota metodológica:** solo existen ocho experimentos reales pareados. Por ello, las pruebas estadísticas se reportan, pero su potencia es limitada. Las asociaciones con la calidad de la señal se analizan a nivel de escenario (`n = 4`) y deben interpretarse como exploratorias.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    friedmanchisquare,
    rankdata,
    spearmanr,
    wilcoxon,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

## 1. Rutas y carga de archivos

In [ ]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Proyecto_PINN_HRF"
)

SUBJECT_ID = "100206"

REAL_BATCH_DIR = (
    PROJECT_ROOT
    / "results"
    / "real"
    / SUBJECT_ID
    / "models_batch"
)

REAL_PREPROCESSING_DIR = (
    PROJECT_ROOT
    / "results"
    / "real"
    / SUBJECT_ID
    / "preprocessing"
)

SYNTHETIC_COMPARISON_DIR = (
    PROJECT_ROOT
    / "results"
    / "synthetic"
    / "comparison"
)

FINAL_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "final_analysis"
)

FINAL_TABLES_DIR = (
    FINAL_RESULTS_DIR
    / "tables"
)

FINAL_FIGURES_DIR = (
    FINAL_RESULTS_DIR
    / "figures"
)

for directory in [
    FINAL_RESULTS_DIR,
    FINAL_TABLES_DIR,
    FINAL_FIGURES_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

PATHS = {
    "real_metrics": (
        REAL_BATCH_DIR
        / "real_model_metrics.csv"
    ),
    "real_summary": (
        REAL_BATCH_DIR
        / "real_model_summary.csv"
    ),
    "parameter_consistency": (
        REAL_BATCH_DIR
        / "real_pinn_parameter_consistency.csv"
    ),
    "hrf_consistency": (
        REAL_BATCH_DIR
        / "real_pinn_hrf_consistency.csv"
    ),
    "quality_summary": (
        REAL_PREPROCESSING_DIR
        / "real_quality_summary.csv"
    ),
    "synthetic_summary": (
        SYNTHETIC_COMPARISON_DIR
        / "synthetic_model_summary.csv"
    ),
}

for name, path in PATHS.items():
    print(
        f"{name:24s}",
        path.exists(),
        path,
    )

In [ ]:
required_paths = [
    PATHS["real_metrics"],
    PATHS["parameter_consistency"],
    PATHS["hrf_consistency"],
    PATHS["quality_summary"],
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Faltan archivos requeridos:\n"
        + "\n".join(
            str(path)
            for path in missing_paths
        )
    )

real_metrics = pd.read_csv(
    PATHS["real_metrics"]
)

parameter_consistency = pd.read_csv(
    PATHS["parameter_consistency"]
)

hrf_consistency = pd.read_csv(
    PATHS["hrf_consistency"]
)

quality_summary = pd.read_csv(
    PATHS["quality_summary"]
)

if PATHS["real_summary"].exists():
    stored_real_summary = pd.read_csv(
        PATHS["real_summary"]
    )
else:
    stored_real_summary = None

if PATHS["synthetic_summary"].exists():
    synthetic_summary = pd.read_csv(
        PATHS["synthetic_summary"]
    )
else:
    synthetic_summary = None

print("Métricas reales:", real_metrics.shape)
print("Consistencia parámetros:", parameter_consistency.shape)
print("Consistencia HRF:", hrf_consistency.shape)
print("Control de calidad:", quality_summary.shape)

## 2. Validación de integridad

In [ ]:
KEY_COLUMNS = [
    "scenario",
    "train_block",
    "test_block",
]

EXPECTED_MODELS = {
    "GLM",
    "MLP",
    "PINN",
}

if len(real_metrics) != 24:
    raise ValueError(
        "Se esperaban 24 filas: "
        "8 experimentos × 3 modelos."
    )

models_found = set(
    real_metrics["model"].unique()
)

if models_found != EXPECTED_MODELS:
    raise ValueError(
        f"Modelos inesperados: {models_found}"
    )

duplicate_count = real_metrics.duplicated(
    KEY_COLUMNS + ["model"]
).sum()

if duplicate_count != 0:
    raise ValueError(
        f"Existen {duplicate_count} filas duplicadas."
    )

model_count_by_experiment = (
    real_metrics
    .groupby(KEY_COLUMNS)["model"]
    .nunique()
)

if not (
    model_count_by_experiment == 3
).all():
    raise ValueError(
        "No todos los experimentos incluyen "
        "los tres modelos."
    )

print(
    "Validación correcta: "
    "8 experimentos pareados y 3 modelos."
)

## 3. Resumen descriptivo real

In [ ]:
real_summary = (
    real_metrics
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        n_experiments=(
            "test_rmse_percent",
            "size",
        ),
        test_rmse_mean=(
            "test_rmse_percent",
            "mean",
        ),
        test_rmse_std=(
            "test_rmse_percent",
            "std",
        ),
        test_rmse_median=(
            "test_rmse_percent",
            "median",
        ),
        test_mae_mean=(
            "test_mae_percent",
            "mean",
        ),
        test_r2_mean=(
            "test_r2",
            "mean",
        ),
        test_r2_std=(
            "test_r2",
            "std",
        ),
        test_pearson_mean=(
            "test_pearson_r",
            "mean",
        ),
        test_pearson_std=(
            "test_pearson_r",
            "std",
        ),
        train_rmse_mean=(
            "train_rmse_percent",
            "mean",
        ),
        train_rmse_std=(
            "train_rmse_percent",
            "std",
        ),
    )
)

real_summary[
    "generalization_gap_mean"
] = (
    real_summary["test_rmse_mean"]
    - real_summary["train_rmse_mean"]
)

real_summary = real_summary.sort_values(
    "test_rmse_mean"
).reset_index(drop=True)

display(real_summary)

real_summary.to_csv(
    FINAL_TABLES_DIR
    / "real_model_performance.csv",
    index=False,
)

In [ ]:
if stored_real_summary is not None:
    comparison_columns = [
        "model",
        "test_rmse_mean",
        "test_r2_mean",
        "test_pearson_mean",
        "train_rmse_mean",
    ]

    verification = (
        real_summary[
            comparison_columns
        ]
        .merge(
            stored_real_summary[
                comparison_columns
            ],
            on="model",
            suffixes=(
                "_recomputed",
                "_stored",
            ),
        )
    )

    for column in comparison_columns[1:]:
        verification[
            f"{column}_difference"
        ] = (
            verification[
                f"{column}_recomputed"
            ]
            - verification[
                f"{column}_stored"
            ]
        )

    display(verification)

## 4. Tabla por experimento, ganadores y generalización

In [ ]:
wide_rmse = real_metrics.pivot(
    index=KEY_COLUMNS,
    columns="model",
    values="test_rmse_percent",
)

wide_r2 = real_metrics.pivot(
    index=KEY_COLUMNS,
    columns="model",
    values="test_r2",
)

wide_pearson = real_metrics.pivot(
    index=KEY_COLUMNS,
    columns="model",
    values="test_pearson_r",
)

experiment_table = (
    wide_rmse
    .add_prefix("rmse_")
    .join(
        wide_r2.add_prefix("r2_")
    )
    .join(
        wide_pearson.add_prefix(
            "pearson_"
        )
    )
    .reset_index()
)

experiment_table["winner_rmse"] = (
    wide_rmse.idxmin(axis=1).to_numpy()
)

experiment_table[
    "best_rmse_percent"
] = (
    wide_rmse.min(axis=1).to_numpy()
)

experiment_table[
    "pinn_improvement_vs_glm_percent"
] = (
    100.0
    * (
        experiment_table["rmse_GLM"]
        - experiment_table["rmse_PINN"]
    )
    / experiment_table["rmse_GLM"]
)

experiment_table[
    "pinn_improvement_vs_mlp_percent"
] = (
    100.0
    * (
        experiment_table["rmse_MLP"]
        - experiment_table["rmse_PINN"]
    )
    / experiment_table["rmse_MLP"]
)

display(experiment_table)

experiment_table.to_csv(
    FINAL_TABLES_DIR
    / "real_per_experiment_results.csv",
    index=False,
)

In [ ]:
win_counts = (
    experiment_table[
        "winner_rmse"
    ]
    .value_counts()
    .rename_axis("model")
    .reset_index(name="wins")
)

win_counts["win_fraction"] = (
    win_counts["wins"]
    / len(experiment_table)
)

display(win_counts)

win_counts.to_csv(
    FINAL_TABLES_DIR
    / "real_model_wins.csv",
    index=False,
)

In [ ]:
real_metrics = real_metrics.copy()

real_metrics[
    "generalization_gap_percent"
] = (
    real_metrics[
        "test_rmse_percent"
    ]
    - real_metrics[
        "train_rmse_percent"
    ]
)

generalization_summary = (
    real_metrics
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        gap_mean=(
            "generalization_gap_percent",
            "mean",
        ),
        gap_std=(
            "generalization_gap_percent",
            "std",
        ),
        gap_median=(
            "generalization_gap_percent",
            "median",
        ),
        gap_min=(
            "generalization_gap_percent",
            "min",
        ),
        gap_max=(
            "generalization_gap_percent",
            "max",
        ),
    )
    .sort_values("gap_mean")
)

display(generalization_summary)

generalization_summary.to_csv(
    FINAL_TABLES_DIR
    / "real_generalization_gap.csv",
    index=False,
)

## 5. Pruebas pareadas

La métrica primaria es el RMSE fuera de muestra. También se reportan MAE, \(R^2\) y correlación de Pearson como métricas secundarias.

In [ ]:
def holm_correction(
    p_values: list[float],
) -> np.ndarray:
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    n_tests = len(p_values)
    order = np.argsort(p_values)

    adjusted_sorted = np.empty(
        n_tests,
        dtype=float,
    )

    running_maximum = 0.0

    for rank, original_index in enumerate(
        order
    ):
        candidate = (
            (n_tests - rank)
            * p_values[original_index]
        )

        running_maximum = max(
            running_maximum,
            candidate,
        )

        adjusted_sorted[rank] = min(
            running_maximum,
            1.0,
        )

    adjusted = np.empty(
        n_tests,
        dtype=float,
    )

    adjusted[order] = adjusted_sorted

    return adjusted


def rank_biserial_paired(
    first: np.ndarray,
    second: np.ndarray,
) -> float:
    differences = (
        np.asarray(first, dtype=float)
        - np.asarray(second, dtype=float)
    )

    differences = differences[
        ~np.isclose(
            differences,
            0.0,
        )
    ]

    if len(differences) == 0:
        return 0.0

    ranks = rankdata(
        np.abs(differences)
    )

    positive_sum = np.sum(
        ranks[differences > 0]
    )

    negative_sum = np.sum(
        ranks[differences < 0]
    )

    return float(
        (
            positive_sum
            - negative_sum
        )
        / (
            positive_sum
            + negative_sum
        )
    )

In [ ]:
METRICS_TO_TEST = {
    "test_rmse_percent": "lower",
    "test_mae_percent": "lower",
    "test_r2": "higher",
    "test_pearson_r": "higher",
}

MODEL_PAIRS = [
    ("GLM", "MLP"),
    ("GLM", "PINN"),
    ("MLP", "PINN"),
]

friedman_rows = []
pairwise_rows = []

for metric, preferred_direction in (
    METRICS_TO_TEST.items()
):
    wide = real_metrics.pivot(
        index=KEY_COLUMNS,
        columns="model",
        values=metric,
    ).dropna()

    friedman_result = (
        friedmanchisquare(
            wide["GLM"],
            wide["MLP"],
            wide["PINN"],
        )
    )

    friedman_rows.append(
        {
            "metric": metric,
            "preferred_direction": (
                preferred_direction
            ),
            "n_pairs": len(wide),
            "friedman_statistic": float(
                friedman_result.statistic
            ),
            "p_value": float(
                friedman_result.pvalue
            ),
        }
    )

    temporary_rows = []
    raw_p_values = []

    for first_model, second_model in (
        MODEL_PAIRS
    ):
        first_values = wide[
            first_model
        ].to_numpy(dtype=float)

        second_values = wide[
            second_model
        ].to_numpy(dtype=float)

        result = wilcoxon(
            first_values,
            second_values,
            alternative="two-sided",
            zero_method="wilcox",
            method="auto",
        )

        temporary_rows.append(
            {
                "metric": metric,
                "preferred_direction": (
                    preferred_direction
                ),
                "first_model": (
                    first_model
                ),
                "second_model": (
                    second_model
                ),
                "n_pairs": len(wide),
                "first_mean": float(
                    np.mean(first_values)
                ),
                "second_mean": float(
                    np.mean(second_values)
                ),
                "mean_difference": float(
                    np.mean(
                        first_values
                        - second_values
                    )
                ),
                "median_difference": float(
                    np.median(
                        first_values
                        - second_values
                    )
                ),
                "wilcoxon_statistic": float(
                    result.statistic
                ),
                "p_value_raw": float(
                    result.pvalue
                ),
                "rank_biserial": (
                    rank_biserial_paired(
                        first_values,
                        second_values,
                    )
                ),
            }
        )

        raw_p_values.append(
            result.pvalue
        )

    corrected = holm_correction(
        raw_p_values
    )

    for row, adjusted_p in zip(
        temporary_rows,
        corrected,
    ):
        row["p_value_holm"] = float(
            adjusted_p
        )

        pairwise_rows.append(row)

friedman_results = pd.DataFrame(
    friedman_rows
)

pairwise_results = pd.DataFrame(
    pairwise_rows
)

print("Friedman:")
display(friedman_results)

print("Wilcoxon pareado con corrección de Holm:")
display(pairwise_results)

friedman_results.to_csv(
    FINAL_TABLES_DIR
    / "real_friedman_tests.csv",
    index=False,
)

pairwise_results.to_csv(
    FINAL_TABLES_DIR
    / "real_pairwise_wilcoxon_holm.csv",
    index=False,
)

## 6. Relación exploratoria entre calidad de los bloques y error

La calidad se define a nivel de escenario, mientras que existen dos direcciones de evaluación por escenario. Para evitar pseudorreplicación, primero se promedia el RMSE de ambas direcciones. Las asociaciones se calculan con solo cuatro escenarios y no deben interpretarse como evidencia inferencial.

In [ ]:
scenario_model_rmse = (
    real_metrics
    .groupby(
        [
            "scenario",
            "model",
        ],
        as_index=False,
    )
    .agg(
        mean_bidirectional_rmse_percent=(
            "test_rmse_percent",
            "mean",
        ),
        directional_rmse_std_percent=(
            "test_rmse_percent",
            "std",
        ),
        mean_bidirectional_r2=(
            "test_r2",
            "mean",
        ),
        mean_bidirectional_pearson=(
            "test_pearson_r",
            "mean",
        ),
    )
)

quality_columns = [
    "scenario",
    "zero_lag_correlation",
    "best_lag_correlation",
    "best_lag_seconds",
    "response_to_noise_ratio",
    "quality_label",
]

quality_error_table = (
    scenario_model_rmse
    .merge(
        quality_summary[
            quality_columns
        ],
        on="scenario",
        how="left",
        validate="many_to_one",
    )
)

if quality_error_table[
    "best_lag_correlation"
].isna().any():
    raise ValueError(
        "No fue posible asociar la calidad "
        "a todos los escenarios."
    )

display(quality_error_table)

quality_error_table.to_csv(
    FINAL_TABLES_DIR
    / "real_quality_error_table.csv",
    index=False,
)

In [ ]:
association_rows = []

QUALITY_VARIABLES = [
    "zero_lag_correlation",
    "best_lag_correlation",
    "response_to_noise_ratio",
]

for model_name in [
    "GLM",
    "MLP",
    "PINN",
]:
    selected = quality_error_table.loc[
        quality_error_table["model"]
        == model_name
    ].copy()

    for quality_variable in (
        QUALITY_VARIABLES
    ):
        result = spearmanr(
            selected[
                quality_variable
            ],
            selected[
                "mean_bidirectional_rmse_percent"
            ],
        )

        association_rows.append(
            {
                "model": model_name,
                "quality_variable": (
                    quality_variable
                ),
                "n_scenarios": len(
                    selected
                ),
                "spearman_rho": float(
                    result.statistic
                ),
                "p_value_exploratory": float(
                    result.pvalue
                ),
            }
        )

quality_associations = pd.DataFrame(
    association_rows
)

display(quality_associations)

quality_associations.to_csv(
    FINAL_TABLES_DIR
    / "real_quality_error_spearman_exploratory.csv",
    index=False,
)

## 7. Consistencia bidireccional de parámetros PINN

In [ ]:
parameter_summary = pd.DataFrame(
    {
        "parameter": [
            "epsilon",
            "tau",
            "alpha",
        ],
        "mean_symmetric_difference_percent": [
            parameter_consistency[
                "epsilon_symmetric_difference_percent"
            ].mean(),
            parameter_consistency[
                "tau_symmetric_difference_percent"
            ].mean(),
            parameter_consistency[
                "alpha_symmetric_difference_percent"
            ].mean(),
        ],
        "median_symmetric_difference_percent": [
            parameter_consistency[
                "epsilon_symmetric_difference_percent"
            ].median(),
            parameter_consistency[
                "tau_symmetric_difference_percent"
            ].median(),
            parameter_consistency[
                "alpha_symmetric_difference_percent"
            ].median(),
        ],
        "std_symmetric_difference_percent": [
            parameter_consistency[
                "epsilon_symmetric_difference_percent"
            ].std(),
            parameter_consistency[
                "tau_symmetric_difference_percent"
            ].std(),
            parameter_consistency[
                "alpha_symmetric_difference_percent"
            ].std(),
        ],
        "maximum_symmetric_difference_percent": [
            parameter_consistency[
                "epsilon_symmetric_difference_percent"
            ].max(),
            parameter_consistency[
                "tau_symmetric_difference_percent"
            ].max(),
            parameter_consistency[
                "alpha_symmetric_difference_percent"
            ].max(),
        ],
    }
).sort_values(
    "mean_symmetric_difference_percent"
)

print("Resumen:")
display(parameter_summary)

print("Detalle por escenario:")
display(parameter_consistency)

parameter_summary.to_csv(
    FINAL_TABLES_DIR
    / "real_pinn_parameter_consistency_summary.csv",
    index=False,
)

parameter_consistency.to_csv(
    FINAL_TABLES_DIR
    / "real_pinn_parameter_consistency_detail.csv",
    index=False,
)

## 8. Consistencia bidireccional de la HRF PINN

In [ ]:
hrf_summary = pd.DataFrame(
    [
        {
            "n_scenarios": len(
                hrf_consistency
            ),
            "hrf_rmse_mean_percent": (
                hrf_consistency[
                    "hrf_rmse_percent"
                ].mean()
            ),
            "hrf_rmse_std_percent": (
                hrf_consistency[
                    "hrf_rmse_percent"
                ].std()
            ),
            "hrf_pearson_mean": (
                hrf_consistency[
                    "hrf_pearson_r"
                ].mean()
            ),
            "hrf_pearson_std": (
                hrf_consistency[
                    "hrf_pearson_r"
                ].std()
            ),
            "peak_absolute_difference_mean_percent": (
                np.mean(
                    np.abs(
                        hrf_consistency[
                            "peak_block1_percent"
                        ]
                        - hrf_consistency[
                            "peak_block2_percent"
                        ]
                    )
                )
            ),
        }
    ]
)

display(hrf_summary)
display(hrf_consistency)

hrf_summary.to_csv(
    FINAL_TABLES_DIR
    / "real_pinn_hrf_consistency_summary.csv",
    index=False,
)

hrf_consistency.to_csv(
    FINAL_TABLES_DIR
    / "real_pinn_hrf_consistency_detail.csv",
    index=False,
)

## 9. Figuras finales reales

In [ ]:
experiment_order = (
    experiment_table[
        [
            "scenario",
            "train_block",
            "test_block",
        ]
    ]
    .copy()
)

experiment_labels = [
    (
        f"{row.scenario}\n"
        f"B{int(row.train_block)}→"
        f"B{int(row.test_block)}"
    )
    for row in experiment_order.itertuples()
]

x_positions = np.arange(
    len(experiment_labels)
)

bar_width = 0.26

plt.figure(figsize=(15, 6))

for offset, model_name in zip(
    [
        -bar_width,
        0.0,
        bar_width,
    ],
    [
        "GLM",
        "MLP",
        "PINN",
    ],
):
    plt.bar(
        x_positions + offset,
        experiment_table[
            f"rmse_{model_name}"
        ],
        width=bar_width,
        label=model_name,
    )

plt.xticks(
    x_positions,
    experiment_labels,
    rotation=35,
    ha="right",
)

plt.ylabel(
    "RMSE fuera de muestra "
    "[puntos porcentuales BOLD]"
)

plt.title(
    "Comparación de modelos sobre datos fMRI reales"
)

plt.legend()
plt.grid(
    axis="y",
    alpha=0.25,
)

plt.tight_layout()

plt.savefig(
    FINAL_FIGURES_DIR
    / "real_models_rmse_by_experiment.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
plot_gap = (
    generalization_summary
    .set_index("model")
    .loc[
        [
            "GLM",
            "MLP",
            "PINN",
        ]
    ]
    .reset_index()
)

plt.figure(figsize=(8, 5))

plt.bar(
    plot_gap["model"],
    plot_gap["gap_mean"],
    yerr=plot_gap["gap_std"],
    capsize=5,
)

plt.axhline(
    0.0,
    linewidth=0.8,
)

plt.ylabel(
    "Brecha de generalización "
    "[RMSE prueba − RMSE entrenamiento]"
)

plt.title(
    "Brecha de generalización en datos reales"
)

plt.grid(
    axis="y",
    alpha=0.25,
)

plt.tight_layout()

plt.savefig(
    FINAL_FIGURES_DIR
    / "real_generalization_gap.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

for model_name in [
    "GLM",
    "MLP",
    "PINN",
]:
    selected = quality_error_table.loc[
        quality_error_table["model"]
        == model_name
    ]

    plt.scatter(
        selected[
            "best_lag_correlation"
        ],
        selected[
            "mean_bidirectional_rmse_percent"
        ],
        s=70,
        label=model_name,
    )

    for row in selected.itertuples():
        plt.annotate(
            row.scenario.replace(
                "motor_",
                "",
            ),
            (
                row.best_lag_correlation,
                row.mean_bidirectional_rmse_percent,
            ),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )

plt.xlabel(
    "Mejor correlación entre bloques"
)

plt.ylabel(
    "RMSE bidireccional medio "
    "[puntos porcentuales BOLD]"
)

plt.title(
    "Repetibilidad de los bloques y error predictivo"
)

plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    FINAL_FIGURES_DIR
    / "real_repeatability_vs_rmse.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
parameter_plot = (
    parameter_consistency[
        [
            "scenario",
            "epsilon_symmetric_difference_percent",
            "tau_symmetric_difference_percent",
            "alpha_symmetric_difference_percent",
        ]
    ]
    .set_index("scenario")
)

x_positions = np.arange(
    len(parameter_plot)
)

bar_width = 0.26

plt.figure(figsize=(12, 5.5))

for offset, column, label in [
    (
        -bar_width,
        "epsilon_symmetric_difference_percent",
        r"$\epsilon$",
    ),
    (
        0.0,
        "tau_symmetric_difference_percent",
        r"$\tau$",
    ),
    (
        bar_width,
        "alpha_symmetric_difference_percent",
        r"$\alpha$",
    ),
]:
    plt.bar(
        x_positions + offset,
        parameter_plot[column],
        width=bar_width,
        label=label,
    )

plt.xticks(
    x_positions,
    parameter_plot.index,
    rotation=25,
    ha="right",
)

plt.ylabel(
    "Diferencia simétrica entre bloques [%]"
)

plt.title(
    "Consistencia bidireccional de parámetros PINN"
)

plt.legend()
plt.grid(
    axis="y",
    alpha=0.25,
)

plt.tight_layout()

plt.savefig(
    FINAL_FIGURES_DIR
    / "real_pinn_parameter_consistency.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
x_positions = np.arange(
    len(hrf_consistency)
)

plt.figure(figsize=(11, 5.5))

plt.bar(
    x_positions,
    hrf_consistency[
        "hrf_rmse_percent"
    ],
)

plt.xticks(
    x_positions,
    hrf_consistency["scenario"],
    rotation=25,
    ha="right",
)

plt.ylabel(
    "RMSE entre HRF estimadas "
    "[puntos porcentuales BOLD]"
)

plt.title(
    "Consistencia de la HRF entre bloques"
)

plt.grid(
    axis="y",
    alpha=0.25,
)

plt.tight_layout()

plt.savefig(
    FINAL_FIGURES_DIR
    / "real_pinn_hrf_consistency.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 10. Síntesis conjunta sintética–real

In [ ]:
if synthetic_summary is not None:
    synthetic_compact = (
        synthetic_summary
        .groupby(
            "model",
            as_index=False,
        )
        .agg(
            synthetic_test_rmse_mean=(
                "test_rmse_mean",
                "mean",
            ),
            synthetic_test_r2_mean=(
                "test_r2_mean",
                "mean",
            ),
            synthetic_hrf_rmse_mean=(
                "hrf_rmse_mean",
                "mean",
            ),
            synthetic_hrf_r2_mean=(
                "hrf_r2_mean",
                "mean",
            ),
        )
    )

    joint_overview = (
        real_summary[
            [
                "model",
                "test_rmse_mean",
                "test_r2_mean",
                "test_pearson_mean",
                "generalization_gap_mean",
            ]
        ]
        .rename(
            columns={
                "test_rmse_mean":
                "real_test_rmse_mean",
                "test_r2_mean":
                "real_test_r2_mean",
                "test_pearson_mean":
                "real_test_pearson_mean",
                "generalization_gap_mean":
                "real_generalization_gap_mean",
            }
        )
        .merge(
            synthetic_compact,
            on="model",
            how="outer",
        )
    )

    display(joint_overview)

    joint_overview.to_csv(
        FINAL_TABLES_DIR
        / "synthetic_real_overview.csv",
        index=False,
    )

else:
    print(
        "No se encontró synthetic_model_summary.csv. "
        "Se omite la síntesis conjunta."
    )

## 11. Exportar tablas LaTeX

Estas tablas se generan como apoyo para el informe. Luego se revisará el número de decimales, los encabezados y el ancho final dentro de `report/`.

In [ ]:
latex_tables = {
    "real_model_performance.tex": (
        real_summary
    ),
    "real_per_experiment_results.tex": (
        experiment_table
    ),
    "real_friedman_tests.tex": (
        friedman_results
    ),
    "real_pairwise_wilcoxon_holm.tex": (
        pairwise_results
    ),
    "real_pinn_parameter_consistency.tex": (
        parameter_summary
    ),
    "real_pinn_hrf_consistency.tex": (
        hrf_summary
    ),
}

for filename, table in (
    latex_tables.items()
):
    latex_text = table.to_latex(
        index=False,
        float_format="%.4f",
        escape=True,
    )

    output_path = (
        FINAL_TABLES_DIR
        / filename
    )

    output_path.write_text(
        latex_text,
        encoding="utf-8",
    )

    print("Guardado:", output_path)

## 12. Resumen automático de resultados

In [ ]:
best_model = (
    real_summary.iloc[0]["model"]
)

best_rmse = float(
    real_summary.iloc[0][
        "test_rmse_mean"
    ]
)

win_dictionary = dict(
    zip(
        win_counts["model"],
        win_counts["wins"],
    )
)

rmse_friedman = (
    friedman_results.loc[
        friedman_results["metric"]
        == "test_rmse_percent"
    ]
    .iloc[0]
)

pearson_friedman = (
    friedman_results.loc[
        friedman_results["metric"]
        == "test_pearson_r"
    ]
    .iloc[0]
)

print(
    "RESUMEN FINAL\n"
    "-------------"
)

print(
    f"Mejor RMSE medio real: "
    f"{best_model} = {best_rmse:.4f} % BOLD"
)

print(
    "Ganadores por RMSE:",
    win_dictionary,
)

print(
    f"Friedman RMSE: "
    f"χ²={rmse_friedman['friedman_statistic']:.3f}, "
    f"p={rmse_friedman['p_value']:.4f}"
)

print(
    f"Friedman Pearson: "
    f"χ²={pearson_friedman['friedman_statistic']:.3f}, "
    f"p={pearson_friedman['p_value']:.4f}"
)

print(
    "Diferencia media simétrica de parámetros:"
)

for row in parameter_summary.itertuples():
    print(
        f"  {row.parameter}: "
        f"{row.mean_symmetric_difference_percent:.2f} %"
    )

print(
    f"Consistencia HRF: "
    f"r medio="
    f"{hrf_summary.iloc[0]['hrf_pearson_mean']:.3f}, "
    f"RMSE medio="
    f"{hrf_summary.iloc[0]['hrf_rmse_mean_percent']:.4f} % BOLD"
)

print(
    "\nResultados guardados en:"
)

print(FINAL_RESULTS_DIR)